# Simple LLM Q&A Workflow with LangGraph (Using Free Gemini API)

This notebook demonstrates a simple LLM workflow built with **LangGraph** using Google's **Gemini API** (Free tier from Google AI Studio).

### Workflow Structure:
$$\text{START} \longrightarrow \text{LLM\_QA} \longrightarrow \text{END}$$

### What happens in `LLM_QA`:
1. Reads the initial `question` from the state.
2. Creates a prompt to answer the question.
3. Calls Google Gemini via `langchain-google-genai`.
4. Updates and returns the state with `answer`.

## 1. Free Gemini API Setup

> **How to get your free Gemini API key:**
> 1. Go to [Google AI Studio](https://aistudio.google.com/app/apikey).
> 2. Sign in with your Google account.
> 3. Click **Create API Key**, copy it, and paste it below when prompted (or save it in a `.env` file as `GEMINI_API_KEY=...`).

In [ ]:
import os
import getpass
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()

# Check for GEMINI_API_KEY or GOOGLE_API_KEY
if not (os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")):
    api_key = getpass.getpass("Enter your free Gemini API key: ")
    os.environ["GEMINI_API_KEY"] = api_key
    os.environ["GOOGLE_API_KEY"] = api_key

print("Gemini API Key configured successfully!")

## 2. Initialize Google Gemini LLM

We import `ChatGoogleGenerativeAI` from `langchain_google_genai` and initialize it with `gemini-2.5-flash` (or `gemini-1.5-flash`).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)
print("Gemini LLM initialized successfully:", llm.model)

## 3. Define the State

The `State` defines the data schema that flows through the graph:
- `question`: the input query
- `answer`: the generated response from the LLM

In [ ]:
from typing import TypedDict

class QAState(TypedDict):
    question: str
    answer: str

## 4. Define the Node: `LLM_QA`

The node function `llm_qa` receives the current `state`, extracts `question`, constructs a prompt, invokes Gemini, and returns the updated state dictionary containing `answer`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Create prompt template
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledgeable and helpful AI assistant. Answer the user question accurately and concisely."),
    ("human", "{question}")
])

# 2. Define node function
def llm_qa(state: QAState) -> dict:
    # Take initial question from state
    question = state["question"]
    
    # Create prompt to answer the question
    prompt_messages = qa_prompt.format_messages(question=question)
    
    # Call the free Gemini LLM
    response = llm.invoke(prompt_messages)
    
    # Output and update into the state
    return {"answer": response.content}

print("Node function llm_qa defined successfully!")

## 5. Build and Compile the Graph

We connect:
$$\text{START} \longrightarrow \text{llm\_qa} \longrightarrow \text{END}$$

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize StateGraph with our State schema
builder = StateGraph(QAState)

# 2. Add node
builder.add_node("llm_qa", llm_qa)

# 3. Add edges: START -> llm_qa -> END
builder.add_edge(START, "llm_qa")
builder.add_edge("llm_qa", END)

# 4. Compile the graph into a runnable app
app = builder.compile()
print("Workflow graph compiled successfully!")

## 6. Visualize the Workflow Graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Mermaid PNG render not available, displaying ASCII graph:")
    print(app.get_graph().draw_ascii())

## 7. Run the Workflow

Provide an initial question in the state and invoke the graph.

In [ ]:
initial_state = {
    "question": "Explain what LangGraph is and how it helps build AI agents in 3 bullet points."
}

result = app.invoke(initial_state)

print("=" * 60)
print("INPUT QUESTION:")
print(result["question"])
print("=" * 60)
print("OUTPUT ANSWER FROM GEMINI:")
print(result["answer"])
print("=" * 60)